## Librerias

In [3]:
x = 2+2

In [2]:
#Liberias
import os
import io
import base64
import boto3
import requests
import openpyxl
import pandas as pd
import numpy as np
from io import BytesIO
from io import StringIO
from dotenv import find_dotenv, load_dotenv

ModuleNotFoundError: No module named 'requests'

# Variables

In [2]:
load_dotenv()
#AWS
region_an_aws = os.getenv("REGION_AN_AWS")
access_key_an_aws = os.getenv("ACCESS_KEY_AN_AWS")
secret_access_key_an_aws = os.getenv("SECRET_ACCESS_KEY_AN_AWS")
bucket_an_aws_pro = os.getenv("BUCKET_AN_AWS_PRO")
temp_session_token_aws_id= os.getenv("TEMP_SESSION_TOKEN_AWS_ID") #solo local

# Funciones

In [3]:
#Funciones S3, para este proyecto dtype="object", na_filter=False,sep=";"
def read_csv_files_aws_s3(prefix, bucket_name, region_name, aws_access_key_id, aws_secret_access_key, separator, na_filter= None, dtype=None):
    if na_filter is None:
        na_filter = True
    session = boto3.Session(
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=temp_session_token_aws_id, #solo local
        region_name=region_name)
    s3 = session.resource("s3")
    bucket = s3.Bucket(bucket_name)
    prefix_objs = bucket.objects.filter(Prefix=prefix)
    prefix_df = []
    for obj in prefix_objs:
        key = obj.key
        body = obj.get()["Body"].read()
        df = pd.read_csv(io.BytesIO(body), encoding="utf8", sep=separator, na_filter=na_filter, dtype=dtype) 
        prefix_df.append(df)
    return pd.concat(prefix_df)

def write_csv_file_aws_s3(data_frame, path_aws_s3, bucket_name, region_name, aws_access_key_id, aws_secret_access_key, separator):
    df = data_frame
    path = path_aws_s3
    s3 = boto3.client("s3",
                      aws_access_key_id=aws_access_key_id,
                      aws_secret_access_key=aws_secret_access_key,
                      aws_session_token=temp_session_token_aws_id, #solo local
                      region_name=region_name)
    
    csv_buf = StringIO()
    df.to_csv(csv_buf, sep=separator, header=True, index=False)
    csv_buf.seek(0)
    s3.put_object(Bucket=bucket_name, Body=csv_buf.getvalue(), Key=path)

# Procesamiento

In [21]:
def products():
    path_s3 = "diners-report-prod/base_dv.csv"
    result = read_csv_files_aws_s3(path_s3, bucket_an_aws_pro,region_an_aws,access_key_an_aws,
                        secret_access_key_an_aws,separator=";", na_filter=False, dtype="object")
    return result

In [22]:

def etl():
    result = products()
    result = result.loc[result["fecha_activacion"].notna()]
    result = result[["identificacion","enrollment_date","balance_miles","estado","rezagados"]]  
    return result

In [23]:
def export():
    result = etl()
    path_s3="diners-report-prod/base_dv_error.csv"
    write_csv_file_aws_s3(result,path_s3,bucket_an_aws_pro,region_an_aws,access_key_an_aws,
                        secret_access_key_an_aws,separator=",")
    print ("Base Exportada ok")

In [24]:
x = export()

Base Exportada ok
